# 01 · Data Preparation

**Purpose:** Pull NFL play-by-play data (2000–2025) via nflfastR, extract field goal attempts,
PATs, and 4th-down non-attempts, apply feature engineering, and produce clean datasets
for the propensity and outcome modeling notebooks.

**Inputs:**
- nflfastR PBP data (downloaded/cached automatically)
- `data/leverage/wp_heads_rulepack.rds` (optional — leverage scoring)
- `data/leverage/pat_heads_rulepack.rds` (optional — leverage scoring)

**Outputs:**
- `data/fg_attempts.csv` — FG + PAT attempts with all features
- `data/fg_nonattempts.csv` — 4th-down non-attempts with features
- `data/fg_all.csv` — combined (attempts + non-attempts)
- `data/kicker_by_game.csv` — primary kicker per game/team

**Sections:**
1. Parameters
2. Imports
3. Data Pull & Cache
4. Feature Engineering
5. Leverage Scoring
6. Standardization
7. Outputs

In [1]:
# ============================================================
# 1. Parameters
# ============================================================
PROJECT_ROOT <- sub('[/\\\\][^/\\\\]*$', '', getwd())

data_dir     <- file.path(PROJECT_ROOT, 'data')
leverage_dir <- file.path(PROJECT_ROOT, 'data', 'leverage')
reports_dir  <- file.path(PROJECT_ROOT, 'reports')
config_path  <- file.path(PROJECT_ROOT, 'config', 'params.yaml')

# Season range — update max year each season
SEASONS <- 2000:2025

# Shared parameters (can be overridden by config/params.yaml)
default_params <- list(
  time_knots           = c(60, 120, 300),
  late_flags           = c(120, 60),
  p_clip_min           = 0.02,
  p_clip_max           = 0.98,
  kickable_cap_modeling = 70,
  kickable_cap_audit   = 50,
  dist_knots           = c(28, 38, 48, 58),
  dist_bounds          = c(18, 70)
)

params <- default_params
if (file.exists(config_path)) {
  tryCatch({
    cfg <- yaml::read_yaml(config_path)
    params <- utils::modifyList(params, cfg, keep.null = TRUE)
  }, error = function(e) message('Config load failed, using defaults: ', e$message))
}

set.seed(20240517)
message('PROJECT_ROOT: ', PROJECT_ROOT)
message('SEASONS: ', min(SEASONS), '-', max(SEASONS))

PROJECT_ROOT: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model



SEASONS: 2000-2025



In [2]:
# ============================================================
# 2. Imports
# ============================================================
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'splines', 'splines2', 'pROC', 'yaml', 'rlang',
  'forcats', 'lubridate', 'nflreadr', 'nflfastR', 'glue'
)
installed <- rownames(installed.packages())
for (pkg in dependencies) {
  if (!pkg %in% installed) install.packages(pkg)
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}

# Ensure output directories exist
for (d in c(data_dir, reports_dir)) {
  if (!dir.exists(d)) dir.create(d, recursive = TRUE, showWarnings = FALSE)
}

message('Libraries loaded.')

Warning message:
"package 'nflreadr' was built under R version 4.5.3"


Warning message:
"package 'nflfastR' was built under R version 4.5.3"


Libraries loaded.



## 3. Data Pull & Cache

Uses `nflfastR::load_pbp()` to pull seasons 2000–2025. On first run this downloads
~500 MB; subsequent runs load from the local cache at `data/pbp_raw.rds`.
Delete the cache file to force a refresh.

In [3]:
pbp_cache <- file.path(data_dir, 'pbp_raw.rds')

if (file.exists(pbp_cache)) {
  message('Loading PBP from cache: ', pbp_cache)
  pbp <- readr::read_rds(pbp_cache)
} else {
  message('Downloading PBP for seasons ', min(SEASONS), '-', max(SEASONS), ' ...')
  pbp <- nflfastR::load_pbp(SEASONS)
  readr::write_rds(pbp, pbp_cache, compress = 'gz')
  message('Cached to: ', pbp_cache)
}

message(sprintf('PBP rows: %s | Seasons: %s-%s',
  format(nrow(pbp), big.mark = ','),
  min(pbp$season, na.rm = TRUE),
  max(pbp$season, na.rm = TRUE)
))

Loading PBP from cache: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/data/pbp_raw.rds



PBP rows: 1,233,492 | Seasons: 2000-2025



In [4]:
# ============================================================
# 4a. Previous Play Lag Features
# ============================================================
# Needed for: clock_running flag, iced flag
pbp_prev <- pbp %>%
  arrange(game_id, play_id) %>%
  group_by(game_id) %>%
  mutate(
    prev_play_type      = dplyr::lag(play_type),
    prev_timeout        = as.integer(dplyr::lag(timeout)),
    prev_timeout_team   = dplyr::lag(timeout_team),
    prev_penalty        = as.integer(dplyr::lag(penalty)),
    prev_incomplete     = as.integer(dplyr::lag(incomplete_pass)),
    prev_out_bounds     = as.integer(dplyr::lag(out_of_bounds)),
    prev_gsr            = dplyr::lag(game_seconds_remaining),
    delta_secs          = prev_gsr - game_seconds_remaining,
    prev_end_quarter    = as.integer(!is.na(dplyr::lag(desc)) &
                            stringr::str_detect(dplyr::lag(desc), '(?i)end\\s+quarter')),
    prev_two_min_warn   = as.integer(!is.na(dplyr::lag(desc)) &
                            stringr::str_detect(dplyr::lag(desc), '(?i)two-?minute\\s+warning'))
  ) %>%
  ungroup() %>%
  select(game_id, play_id, prev_play_type, prev_timeout, prev_timeout_team,
         prev_penalty, prev_incomplete, prev_out_bounds,
         prev_end_quarter, prev_two_min_warn, delta_secs)

message('Lag features built: ', nrow(pbp_prev), ' rows')

Lag features built: 1233492 rows



In [5]:
# ============================================================
# 4b. Field Goal & PAT Attempts
# ============================================================
fg_attempts_raw <- pbp %>%
  filter(
    season %in% SEASONS,
    play_type %in% c('field_goal', 'extra_point'),
    (field_goal_result  %in% c('made', 'missed', 'blocked')) |
    (extra_point_result %in% c('good', 'failed', 'blocked'))
  ) %>%
  mutate(
    is_pat           = as.integer(play_type == 'extra_point'),
    kick_result_raw  = dplyr::coalesce(field_goal_result, extra_point_result),
    kick_result      = case_when(
      kick_result_raw %in% c('made', 'good')    ~ 'made',
      kick_result_raw %in% c('missed', 'failed') ~ 'missed',
      kick_result_raw == 'blocked'               ~ 'blocked',
      TRUE ~ NA_character_
    ),
    kick_distance = suppressWarnings(as.numeric(kick_distance)),
    kick_distance = if_else(is_pat == 1L & is.na(kick_distance), 33, kick_distance),
    attempted     = 1L
  ) %>%
  filter(!is.na(kick_distance)) %>%
  transmute(
    game_id, play_id, old_game_id,
    season = as.integer(season), week = as.integer(week), season_type,
    playoffs = as.integer(season_type == 'POST'),
    qtr = as.integer(qtr),
    game_date = as.Date(game_date),
    home_team, away_team, posteam, defteam,
    game_seconds_remaining, quarter_seconds_remaining,
    score_differential, yardline_100, ydstogo,
    wp, wpa, epa,
    posteam_timeouts_remaining, defteam_timeouts_remaining,
    goal_to_go,
    is_ot = qtr >= 5,
    roof, surface, temp, wind, weather,
    stadium_id, stadium, location,
    kick_distance, kicker_player_id, kicker_player_name,
    field_goal_result, extra_point_result,
    kick_result, is_pat, attempted,
    play_type_original = play_type
  )

message(sprintf('FG/PAT attempts: %s (FG: %s, PAT: %s)',
  nrow(fg_attempts_raw),
  sum(fg_attempts_raw$is_pat == 0),
  sum(fg_attempts_raw$is_pat == 1)
))

FG/PAT attempts: 59105 (FG: 26766, PAT: 32339)



In [6]:
# ============================================================
# 4c. 4th-Down Non-Attempt Opportunities
# ============================================================
fg_nonattempts_raw <- pbp %>%
  filter(
    season %in% SEASONS,
    down == 4L,
    !is.na(yardline_100),
    !is.na(ydstogo),
    play_type %in% c('pass', 'run', 'punt', 'qb_kneel', 'qb_spike')
  ) %>%
  mutate(derived_kick_distance = yardline_100 + 17L) %>%
  transmute(
    game_id, play_id, old_game_id,
    season = as.integer(season), week = as.integer(week), season_type,
    playoffs = as.integer(season_type == 'POST'),
    qtr = as.integer(qtr), down,
    game_date = as.Date(game_date),
    home_team, away_team, posteam, defteam,
    game_seconds_remaining, quarter_seconds_remaining,
    score_differential, yardline_100, ydstogo,
    wp, wpa, epa,
    posteam_timeouts_remaining, defteam_timeouts_remaining,
    goal_to_go,
    is_ot = qtr >= 5,
    roof, surface, temp, wind, weather,
    stadium_id, stadium, location,
    kick_distance         = derived_kick_distance,
    kicker_player_id      = NA_character_,
    kicker_player_name    = NA_character_,
    field_goal_result     = NA_character_,
    extra_point_result    = NA_character_,
    kick_result           = NA_character_,
    is_pat                = 0L,
    attempted             = 0L,
    play_type_original    = play_type
  )

message('Non-attempts (4th down): ', nrow(fg_nonattempts_raw))

Non-attempts (4th down): 78563



In [7]:
# ============================================================
# 4d. Combine & Add Situational Features
# ============================================================
fg_all <- bind_rows(fg_attempts_raw, fg_nonattempts_raw) %>%
  left_join(pbp_prev, by = c('game_id', 'play_id')) %>%
  mutate(
    # Late-game flags
    l2m = as.integer(qtr %in% c(2L, 4L) & quarter_seconds_remaining <= 120),

    # Clock running: no stoppage events on prior play
    clock_running = as.integer(
      is_pat == 0L &
      !is.na(delta_secs) & delta_secs > 0 &
      coalesce(prev_timeout,      0L) == 0L &
      coalesce(prev_penalty,      0L) == 0L &
      coalesce(prev_incomplete,   0L) == 0L &
      coalesce(prev_out_bounds,   0L) == 0L &
      coalesce(prev_end_quarter,  0L) == 0L &
      coalesce(prev_two_min_warn, 0L) == 0L
    ),

    # Iced: opponent called timeout immediately before kick
    iced = as.integer(
      is_pat == 0L &
      coalesce(prev_timeout, 0L) == 1L &
      !is.na(prev_timeout_team) &
      prev_timeout_team == defteam
    )
  )

message('Combined fg_all: ', nrow(fg_all), ' rows')

Combined fg_all: 137668 rows



In [8]:
# ============================================================
# 4e. Kicker Age & Experience
# ============================================================
players <- nflreadr::load_players() %>%
  transmute(
    gsis_id      = as.character(gsis_id),
    birth_date   = as.Date(birth_date),
    rookie_season = suppressWarnings(as.integer(rookie_season))
  )

fg_all <- fg_all %>%
  mutate(kicker_player_id = as.character(kicker_player_id)) %>%
  left_join(players, by = c('kicker_player_id' = 'gsis_id')) %>%
  mutate(
    kicker_age = as.numeric(difftime(game_date, birth_date, units = 'days')) / 365.25,
    kicker_experience = if_else(
      !is.na(rookie_season),
      as.numeric(season - rookie_season) + 1,
      NA_real_
    )
  )

message('Kicker age/experience joined.')

Kicker age/experience joined.



In [9]:
# ============================================================
# 4f. Venue Flags (indoor, turf, altitude)
# ============================================================
fg_all <- fg_all %>%
  mutate(
    roof        = forcats::fct_na_value_to_level(as.factor(roof), 'unknown'),
    surface     = forcats::fct_na_value_to_level(as.factor(surface), 'unknown'),
    roof_std    = tolower(as.character(roof)),
    surface_std = tolower(as.character(surface)),
    indoors     = as.integer(roof_std %in% c('dome', 'closed', 'open')),
    is_turf     = as.integer(surface_std != 'grass'),
    high_altitude = as.integer(!is.na(stadium_id) &
                      stringr::str_detect(stadium_id, '^(DEN|MEX)'))
  )

In [10]:
# ============================================================
# 4g. Weather Parsing & Imputation
# ============================================================
fg_all <- fg_all %>%
  mutate(
    weather = if_else(weather == '', NA_character_, weather),
    temp_from_weather  = suppressWarnings(as.numeric(stringr::str_extract(
                           stringr::str_extract(weather, '(?i)temp\\s*:?\\s*(-?\\d{1,3})'), '-?\\d{1,3}'))),
    wind_from_weather  = suppressWarnings(as.numeric(stringr::str_extract(
                           stringr::str_extract(weather, '(?i)(\\d{1,3})\\s*mph'), '\\d{1,3}'))),
    humidity_from_weather = suppressWarnings(as.numeric(stringr::str_extract(
                              stringr::str_extract(weather, '(?i)(\\d{1,3})\\s*%'), '\\d{1,3}'))),
    temp     = dplyr::coalesce(temp,     temp_from_weather),
    wind     = dplyr::coalesce(wind,     wind_from_weather),
    humidity = humidity_from_weather,
    # Indoor defaults
    temp     = if_else(is.na(temp)     & indoors == 1L, 71, temp),
    wind     = if_else(is.na(wind)     & indoors == 1L, 0,  wind),
    humidity = if_else(is.na(humidity) & indoors == 1L, 45, humidity)
  ) %>%
  select(-temp_from_weather, -wind_from_weather, -humidity_from_weather)

# Stadium-month median imputation for outdoor venues
outdoor_medians <- fg_all %>%
  filter(indoors == 0L) %>%
  mutate(month = lubridate::month(game_date)) %>%
  group_by(stadium_id, month) %>%
  summarise(
    temp_med     = median(temp,     na.rm = TRUE),
    wind_med     = median(wind,     na.rm = TRUE),
    humidity_med = median(humidity, na.rm = TRUE),
    .groups = 'drop'
  )

global_temp_med     <- median(fg_all$temp[fg_all$indoors == 0L],     na.rm = TRUE)
global_wind_med     <- median(fg_all$wind[fg_all$indoors == 0L],     na.rm = TRUE)
global_humidity_med <- median(fg_all$humidity[fg_all$indoors == 0L], na.rm = TRUE)

fg_all <- fg_all %>%
  mutate(month = lubridate::month(game_date)) %>%
  left_join(outdoor_medians, by = c('stadium_id', 'month')) %>%
  mutate(
    temp     = dplyr::coalesce(temp,     if_else(indoors == 0L, temp_med,     NA_real_)),
    wind     = dplyr::coalesce(wind,     if_else(indoors == 0L, wind_med,     NA_real_)),
    humidity = dplyr::coalesce(humidity, if_else(indoors == 0L, humidity_med, NA_real_)),
    temp     = dplyr::coalesce(temp,     if_else(indoors == 0L, global_temp_med,     NA_real_)),
    wind     = dplyr::coalesce(wind,     if_else(indoors == 0L, global_wind_med,     NA_real_)),
    humidity = dplyr::coalesce(humidity, if_else(indoors == 0L, global_humidity_med, NA_real_)),
    # Force indoor venues
    temp = case_when(indoors == 1L & roof_std %in% c('dome', 'closed') ~ 71, TRUE ~ temp),
    wind = case_when(
      indoors == 1L & roof_std %in% c('dome', 'closed') ~ 0,
      indoors == 1L & roof_std == 'open' & !is.na(wind) & wind > 5 ~ 5,
      TRUE ~ wind
    )
  ) %>%
  select(-temp_med, -wind_med, -humidity_med, -month)

In [11]:
# ============================================================
# 4h. Weather Condition Flags
# ============================================================
fg_all <- fg_all %>%
  mutate(
    weather_clean = stringr::str_to_lower(coalesce(weather, '')) %>% stringr::str_squish(),
    weather_clean = if_else(weather_clean == '' & indoors == 0L, 'clear', weather_clean),
    no_rain_phrase = as.integer(stringr::str_detect(
      weather_clean, 'no\\s+chance\\s+of\\s+rain|0%\\s*chance')),
    is_snow_sleet = as.integer(
      indoors != 1L & stringr::str_detect(
        weather_clean, '\\bsnow\\b|\\bflurr|\\bsleet\\b|\\bfreezing\\s+rain\\b|\\bwintry\\s+mix\\b')),
    is_rain_showers = as.integer(
      indoors != 1L & stringr::str_detect(
        weather_clean, '\\brain\\b|\\braining\\b|\\bshowers?\\b|\\bdrizzle\\b|\\bstorm\\b')),
    is_rain_showers = if_else(is_snow_sleet == 1L | no_rain_phrase == 1L, 0L, is_rain_showers)
  ) %>%
  select(-no_rain_phrase)

In [12]:
# ============================================================
# 5. Leverage Scoring (conditional on rulepack files)
# ============================================================
fg_pack_path  <- file.path(leverage_dir, 'wp_heads_rulepack.rds')
pat_pack_path <- file.path(leverage_dir, 'pat_heads_rulepack.rds')
leverage_ok   <- file.exists(fg_pack_path) && file.exists(pat_pack_path)

if (!leverage_ok) {
  message('Leverage rulepacks not found — skipping leverage scoring. ',
          'Expected in: ', leverage_dir)
  lev_cols <- c('wp_make_hat', 'wp_miss_hat', 'leverage',
                'wp_make_hat_rules', 'wp_miss_hat_rules', 'leverage_rules')
  for (col in lev_cols) fg_all[[col]] <- NA_real_
} else {
  fg_pack  <- tryCatch(readRDS(fg_pack_path), error = function(e) {
    message('Error loading fg_pack: ', e$message)
    NULL
  })
  pat_pack <- tryCatch(readRDS(pat_pack_path), error = function(e) {
    message('Error loading pat_pack: ', e$message)
    NULL
  })

  if (is.null(fg_pack) || is.null(pat_pack)) {
    message('Failed to load valid rulepacks. Skipping leverage scoring.')
    lev_cols <- c('wp_make_hat', 'wp_miss_hat', 'leverage',
                  'wp_make_hat_rules', 'wp_miss_hat_rules', 'leverage_rules')
    for (col in lev_cols) fg_all[[col]] <- NA_real_
  } else {
    eps <- 1e-6
    clamp01 <- function(x) pmin(pmax(x, eps), 1 - eps)

    desired_cols <- c('.row_id', 'wp_make_hat', 'wp_miss_hat', 'leverage',
                      'wp_make_hat_rules', 'wp_miss_hat_rules', 'leverage_rules')

    fg_all <- fg_all %>% mutate(.row_id = dplyr::row_number())

    # Score non-PAT regulation plays
    fg_nonpat_reg <- fg_all %>% filter(is_pat == 0L, !is_ot, !is.na(yardline_100))
    if (nrow(fg_nonpat_reg) > 0) {
      scored_reg <- tryCatch({
        sr <- fg_nonpat_reg %>%
          mutate(
            wp_make_hat = clamp01(predict(fg_pack$mod_make,
                           newdata = fg_pack$mk_feats_make(fg_nonpat_reg), type = 'response')),
            wp_miss_hat = clamp01(predict(fg_pack$mod_miss,
                           newdata = fg_pack$mk_feats_miss(fg_nonpat_reg), type = 'response')),
            leverage    = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1)
          )
        sr <- fg_pack$apply_endgame_rules(sr)
        sr %>% dplyr::select(dplyr::any_of(desired_cols))
      }, error = function(e) {
        message('Error scoring non-PAT regulation plays: ', e$message)
        tibble::tibble(.row_id = integer(),
          wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
          wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
          leverage_rules = numeric())
      })
    } else {
      scored_reg <- tibble::tibble(.row_id = integer(),
        wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
        wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
        leverage_rules = numeric())
    }

    # OT constant for non-PAT OT plays
    ot_const <- scored_reg %>%
      summarise(v = median(leverage_rules, na.rm = TRUE)) %>% pull(v)
    if (is.na(ot_const)) ot_const <- 0.5

    scored_ot <- fg_all %>% filter(is_pat == 0L, is_ot) %>%
      transmute(.row_id,
        wp_make_hat = NA_real_, wp_miss_hat = NA_real_,
        leverage = ot_const, wp_make_hat_rules = NA_real_,
        wp_miss_hat_rules = NA_real_,
        leverage_rules = ot_const)

    # PATs
    pats_reg <- fg_all %>% filter(is_pat == 1L, !is_ot)
    scored_pat <- tryCatch({
      if (nrow(pats_reg) > 0) {
        sp <- pats_reg %>%
          mutate(
            wp_make_hat = clamp01(predict(pat_pack$mod_pat_make,
                           newdata = pat_pack$mk_feats_pat_make(pats_reg), type = 'response')),
            wp_miss_hat = clamp01(predict(pat_pack$mod_pat_miss,
                           newdata = pat_pack$mk_feats_pat_miss(pats_reg), type = 'response')),
            leverage    = pmin(pmax(wp_make_hat - wp_miss_hat, 0), 1)
          )
        sp <- pat_pack$apply_pat_rules_min(sp)
        sp %>% dplyr::select(dplyr::any_of(desired_cols))
      } else {
        tibble::tibble(.row_id = integer(),
          wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
          wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
          leverage_rules = numeric())
      }
    }, error = function(e) {
      message('Error scoring PATs: ', e$message)
      tibble::tibble(.row_id = integer(),
        wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
        wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
        leverage_rules = numeric())
    })

    scored_pat_ot <- tryCatch({
      spo <- fg_all %>% filter(is_pat == 1L, is_ot)
      if (nrow(spo) > 0) {
        spo <- pat_pack$pat_ot_const(spo)
        spo %>% dplyr::select(dplyr::any_of(desired_cols))
      } else {
        tibble::tibble(.row_id = integer(),
          wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
          wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
          leverage_rules = numeric())
      }
    }, error = function(e) {
      message('Error scoring PAT OT plays: ', e$message)
      tibble::tibble(.row_id = integer(),
        wp_make_hat = numeric(), wp_miss_hat = numeric(), leverage = numeric(),
        wp_make_hat_rules = numeric(), wp_miss_hat_rules = numeric(),
        leverage_rules = numeric())
    })

    scored_all <- bind_rows(scored_reg, scored_ot, scored_pat, scored_pat_ot)
    fg_all <- fg_all %>%
      left_join(scored_all, by = '.row_id') %>%
      select(-.row_id)

    message('Leverage scored.')
  }
}

Error scoring PAT OT plays: attempt to apply non-function



Leverage scored.



In [13]:
# ============================================================
# 5b. Implausible Wind Value Correction
# ============================================================
# The wind field is parsed from free-text game weather strings and contains a
# handful of implausible readings that are almost certainly parsing errors
# rather than real gusts -- e.g. 71 mph in one 2016 game and 70 mph in one
# 2008 game, against a physically reasonable NFL sideline maximum. Unlike a
# statistical percentile cap (which would also touch genuinely windy but
# plausible games), we correct only values that fail a fixed physical
# plausibility threshold.
#
# Corrected rows are treated as missing (not clipped to the threshold, which
# would assert a specific, unverified wind speed) and re-imputed with the
# median wind among outdoor attempted kicks below the threshold -- the same
# reference population used for standardization below.
WIND_IMPLAUSIBLE_MPH <- 40

wind_bad <- !is.na(fg_all$wind) & fg_all$wind > WIND_IMPLAUSIBLE_MPH
wind_fallback_outdoor <- median(
  fg_all$wind[fg_all$attempted == 1L & fg_all$indoors == 0L &
              !is.na(fg_all$wind) & fg_all$wind <= WIND_IMPLAUSIBLE_MPH],
  na.rm = TRUE
)

wind_bad_summary <- fg_all %>%
  dplyr::filter(wind_bad) %>%
  dplyr::mutate(kind = dplyr::case_when(
    is_pat == 1L ~ 'PAT', attempted == 1L ~ 'FG attempt', TRUE ~ 'non-attempt')) %>%
  dplyr::count(game_id, wind, kind)

fg_all <- fg_all %>%
  mutate(
    wind = dplyr::case_when(
      wind_bad & indoors == 1L ~ 0,
      wind_bad                 ~ wind_fallback_outdoor,
      TRUE                     ~ wind
    )
  )

message(sprintf(
  'Wind: %d of %d rows exceeded %d mph (implausible, %d distinct games) and were replaced with the outdoor reference median (%.1f mph).',
  sum(wind_bad), nrow(fg_all), WIND_IMPLAUSIBLE_MPH,
  dplyr::n_distinct(wind_bad_summary$game_id), wind_fallback_outdoor
))
print(wind_bad_summary)

Wind: 58 of 137668 rows exceeded 40 mph (implausible, 3 distinct games) and were replaced with the outdoor reference median (8.0 mph).



── nflverse play by play data ──────────────────────────────────────────────────



ℹ Data updated: 2026-02-12 04:57:42 EST



# A tibble: 9 × 4
  game_id          wind kind            n
  <chr>           <dbl> <chr>       <int>
1 2008_02_TEN_CIN    70 FG attempt      2
2 2008_02_TEN_CIN    70 PAT             4
3 2008_02_TEN_CIN    70 non-attempt    17
4 2016_13_NYG_PIT    71 FG attempt      3
5 2016_13_NYG_PIT    71 PAT             3
6 2016_13_NYG_PIT    71 non-attempt    12
7 2023_13_SF_PHI     44 FG attempt      2
8 2023_13_SF_PHI     44 PAT             7
9 2023_13_SF_PHI     44 non-attempt     8


In [14]:
# ============================================================
# 6. Z-Score Standardization & Modeling Flags
# ============================================================
# Standardization stats computed on ATTEMPTS ONLY to avoid data leakage
stats_attempts <- fg_all %>%
  filter(attempted == 1L) %>%
  summarise(
    wind_mean = mean(wind, na.rm = TRUE), wind_sd = sd(wind, na.rm = TRUE),
    temp_mean = mean(temp, na.rm = TRUE), temp_sd = sd(temp, na.rm = TRUE),
    hum_mean  = mean(humidity, na.rm = TRUE), hum_sd = sd(humidity, na.rm = TRUE),
    age_mean  = mean(kicker_age, na.rm = TRUE), age_sd = sd(kicker_age, na.rm = TRUE),
    exp_mean  = mean(kicker_experience, na.rm = TRUE), exp_sd = sd(kicker_experience, na.rm = TRUE),
    lev_mean  = mean(leverage_rules, na.rm = TRUE), lev_sd = sd(leverage_rules, na.rm = TRUE)
  )

safe_z <- function(x, m, s) if_else(!is.na(x) & !is.na(s) & s > 0, (x - m) / s, NA_real_)

fg_all <- fg_all %>%
  mutate(
    wind_z     = safe_z(wind,     stats_attempts$wind_mean, stats_attempts$wind_sd),
    temp_z     = safe_z(temp,     stats_attempts$temp_mean, stats_attempts$temp_sd),
    humidity_z = safe_z(humidity, stats_attempts$hum_mean,  stats_attempts$hum_sd),
    kicker_age_z = safe_z(kicker_age, stats_attempts$age_mean, stats_attempts$age_sd),
    kicker_experience_z = safe_z(kicker_experience,
                           stats_attempts$exp_mean, stats_attempts$exp_sd),
    leverage_z = safe_z(leverage_rules, stats_attempts$lev_mean, stats_attempts$lev_sd),
    kick_made  = if_else(attempted == 1L & !is.na(kick_result),
                         as.integer(kick_result == 'made'), NA_integer_),
    go_ahead   = as.integer(score_differential == 0),
    to_tie     = as.integer(score_differential == -3),
    one_score  = as.integer(!is.na(score_differential) & abs(score_differential) <= 8)
  )

message('Z-scores and flags added.')

Z-scores and flags added.



In [15]:
# ============================================================
# 7. Kicker-by-Game Mapping Table
# ============================================================
# Maps primary kicker per game/team — used in notebook 03 to impute
# kicker IDs for non-attempt plays.
kicker_by_game <- fg_all %>%
  filter(is_pat == 0L, attempted == 1L, !is.na(kicker_player_id)) %>%
  group_by(game_id, posteam) %>%
  summarise(
    kicker_player_id = names(which.max(table(kicker_player_id))),
    .groups = 'drop'
  ) %>%
  rename(team = posteam)

message('kicker_by_game: ', nrow(kicker_by_game), ' rows')

kicker_by_game: 12184 rows



In [16]:
# ============================================================
# 8. Split & QA
# ============================================================
fg_attempts  <- fg_all %>% filter(attempted == 1L) %>% arrange(season, week, game_id, play_id)
fg_nonattempts <- fg_all %>% filter(attempted == 0L) %>% arrange(season, week, game_id, play_id)

# QA summary
qa <- dplyr::bind_rows(
  fg_attempts %>% summarise(
    dataset = 'fg_attempts', n = n(), pats = sum(is_pat == 1L),
    min_dist = min(kick_distance, na.rm = TRUE),
    max_dist = max(kick_distance, na.rm = TRUE),
    miss_temp = sum(is.na(temp)), miss_wind = sum(is.na(wind)),
    seasons = paste(range(season), collapse = '-')),
  fg_nonattempts %>% summarise(
    dataset = 'fg_nonattempts', n = n(), pats = 0L,
    min_dist = min(kick_distance, na.rm = TRUE),
    max_dist = max(kick_distance, na.rm = TRUE),
    miss_temp = sum(is.na(temp)), miss_wind = sum(is.na(wind)),
    seasons = paste(range(season), collapse = '-'))
)
print(qa)

# A tibble: 2 × 8
  dataset            n  pats min_dist max_dist miss_temp miss_wind seasons  
  <chr>          <int> <int>    <dbl>    <dbl>     <int>     <int> <chr>    
1 fg_attempts    59105 32339       18       76         0         0 2000-2025
2 fg_nonattempts 78563     0       18      116         0         0 2000-2025


In [17]:
# ============================================================
# 9. Persist Outputs
# ============================================================
readr::write_csv(fg_attempts,    file.path(data_dir, 'fg_attempts.csv'))
readr::write_csv(fg_nonattempts, file.path(data_dir, 'fg_nonattempts.csv'))
readr::write_csv(fg_all,         file.path(data_dir, 'fg_all.csv'))
readr::write_csv(kicker_by_game, file.path(data_dir, 'kicker_by_game.csv'))

# Schema preview for reports
schema <- tibble::tibble(
  name   = names(fg_all),
  class  = purrr::map_chr(fg_all, ~ paste(class(.x), collapse = '/')),
  sample = purrr::map_chr(fg_all, ~ paste(utils::head(.x, 3), collapse = ', '))
)
readr::write_csv(schema, file.path(reports_dir, 'data_prep_schema_preview.csv'))

message('\n=== Outputs Written ===')
message('fg_attempts.csv:     ', nrow(fg_attempts),    ' rows')
message('fg_nonattempts.csv:  ', nrow(fg_nonattempts), ' rows')
message('fg_all.csv:          ', nrow(fg_all),         ' rows')
message('kicker_by_game.csv:  ', nrow(kicker_by_game), ' rows')

# Session info
readr::write_lines(capture.output(sessionInfo()),
                   file.path(reports_dir, 'session_info.txt'), append = TRUE)


=== Outputs Written ===



fg_attempts.csv:     59105 rows



fg_nonattempts.csv:  78563 rows



fg_all.csv:          137668 rows



kicker_by_game.csv:  12184 rows

